# Visión por Computadora II #

## CEIA 21Co2025 ##

## TP Integrador ##

José Luis Diaz (diazjoseluis@gmail.com)

Ricardo Silvera (rsilvera@thalu.com.ar)

José Aviani (jose.aviani@gmail.com)


---

### Dataset: ###

#### 'X-Ray Baggage Scanner Anomaly Detection' de Kaggle ####

https://www.kaggle.com/datasets/orvile/x-ray-baggage-anomaly-detection/ 

##### Tipo de dataset #####

El conjunto de datos “X-Ray Baggage Scanner Anomaly Detection” de Kaggle contiene imágenes de rayos X de equipaje obtenidas en escáneres de seguridad. Las imágenes muestran valijas y bolsos con objetos superpuestos y densidades distintas. Este es un problema de detección de objetos.

##### Estructura #####

Cada imagen (en formato jpg) está acompañada por un archivo (txt) en formato YOLO, donde se indican las regiones que contienen objetos de interés mediante bounding boxes y un identificador de clase. Estas clases corresponden a 5 objetos potencialmente peligrosos:

“gun” (0), “knife” (1), “pliers” (2), “scissors” (3) y “wrench” (4).

Una misma imagen puede no contener ninguna amenaza, o incluir varias instancias de distintos tipos. Las etiquetas están en formato YOLO: cada .txt contiene 0 o más líneas con

    class_id x_center y_center width height (normalizado 0–1)

correspondientes a los objetos (clases) en la imagen.

El dataset contiene tres folders: “train”, “valid” y “test”. A su vez cada uno de estos folders contierne dos folders: “images” (con las imágenes el formato jpg) y “labels” (los labes en formato txt/YOLO).

---

Importar librerias:

In [ ]:
%pip install gdown
%pip install torch
%pip install torchvision
%pip install matplotlib

In [ ]:
import os
import zipfile
import gdown
import shutil
import glob
import torch
import torchvision
import numpy as np
from collections import Counter
from PIL import Image
import random
import matplotlib.pyplot as plt
from torchvision.ops import box_convert
import pandas as pd
from torch.utils.data import WeightedRandomSampler

In [ ]:
random.seed(42)

---

#### Descargar dataset ####

In [ ]:
data_dir = os.path.join(os.getcwd(), "data")

def descargar_dataset(data_dir):

  # Verificar si exsite la carpeta
  if not os.path.exists(data_dir):

    try:
      # Nombre del archivo ZIP que se va a guardar
      ZIP_NAME = "kaggle-xray_baggage_scanner_anomaly_detection.zip"
      zip_path = os.path.join(data_dir, ZIP_NAME)

      # Crear carpeta data
      os.makedirs(data_dir, exist_ok=True)

      # Descargar el ZIP
      print("Descargando archivo zip ...")
      # Lo tomamos de Google Drive porque Kaggle requeire autenticación
      gdown.download(id="1IqPblTm7nmKFpHXtl4beopE_SajTBoI0", output=zip_path, quiet=False)
      print("Archivo zip descargado.")

      # Descomprimir el ZIP
      print("Descomprimiendo archivo ...")
      with zipfile.ZipFile(zip_path, "r") as zf:
          zf.extractall(data_dir)
      print("Archivo descomprimido.")
    except:
      # En caso de error, eliminar la carpeta creada
      shutil.rmtree(data_dir, ignore_errors=True)
      print("Ocurrió un error al descargar el dataset.")

  print(f"Dataset descargado en: '{data_dir}'")


# Descargar el dataset (solo si no existe la carpeta data)
descargar_dataset(data_dir)

---

#### Cargar dataset ####

In [ ]:
# Dataset YOLO

CLASS_NAMES = ['gun', 'knife', 'pliers', 'scissors', 'wrench']

class YoloDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, dir_name, transforms=None):
        dir_path = os.path.join(data_dir, dir_name)
        images_dir = os.path.join(dir_path, "images")
        labels_dir = os.path.join(dir_path, "labels")
        
        self.images = sorted(glob.glob(os.path.join(images_dir, "*")))
        self.labels_dir = labels_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.images)

    def _read_yolo_txt(self, label_path):
        boxes_cxcywh = []
        labels = []
        if not os.path.exists(label_path):
            return torch.zeros((0,4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
        with open(label_path, "r") as f:
            lines = [ln.strip() for ln in f.readlines() if ln.strip()]
        if not lines:
            return torch.zeros((0,4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
        for ln in lines:
            c, cx, cy, w, h = ln.split()
            labels.append(int(c))
            boxes_cxcywh.append([float(cx), float(cy), float(w), float(h)])
        return torch.tensor(boxes_cxcywh, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(self.labels_dir, base + ".txt")
        boxes_cxcywh_norm, labels = self._read_yolo_txt(lbl_path)

        # Desnormalizar a píxeles
        if boxes_cxcywh_norm.numel() > 0:
            scale = torch.tensor([W, H, W, H], dtype=torch.float32)
            boxes_cxcywh_px = boxes_cxcywh_norm * scale
            boxes_xyxy = box_convert(boxes_cxcywh_px, in_fmt="cxcywh", out_fmt="xyxy")
            # Clampear por seguridad
            boxes_xyxy[:, [0,2]] = boxes_xyxy[:, [0,2]].clamp(0, W)
            boxes_xyxy[:, [1,3]] = boxes_xyxy[:, [1,3]].clamp(0, H)
        else:
            boxes_xyxy = torch.zeros((0,4), dtype=torch.float32)

        target = {
            "boxes": boxes_xyxy,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": ((boxes_xyxy[:,2]-boxes_xyxy[:,0]) * (boxes_xyxy[:,3]-boxes_xyxy[:,1]))
              if boxes_xyxy.numel() else torch.tensor([], dtype=torch.float32),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)


In [ ]:
dataloader_batch_size = 32

transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

train_dataset = YoloDetectionDataset("train", transforms=transforms)
valid_dataset = YoloDetectionDataset("valid", transforms=transforms)
test_dataset = YoloDetectionDataset("test", transforms=transforms)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)

In [ ]:
print (f"Imágenes de train: {len(train_dataloader.dataset)}")
print (f"Imágenes de valid: {len(valid_dataloader.dataset)}")
print (f"Imágenes de test: {len(test_dataloader.dataset)}")

---

#### Validar el dataset ####

Verificamos el dataset:

In [ ]:
def chequeo_consistencia(dir_name):
    dir_path = os.path.join(data_dir, dir_name)
    images_dir = os.path.join(dir_path, "images")
    labels_dir = os.path.join(dir_path, "labels")

    # Listar imágenes y labels
    img_paths = sorted(glob.glob(os.path.join(images_dir, "*")))
    lbl_paths = sorted(glob.glob(os.path.join(labels_dir, "*.txt")))
    
    img_names = {os.path.splitext(os.path.basename(p))[0] for p in img_paths}
    lbl_names = {os.path.splitext(os.path.basename(p))[0] for p in lbl_paths}
    
    # Correspondencias
    imgs_sin_label = img_names - lbl_names     # imágenes sin .txt
    labels_sin_img = lbl_names - img_names     # .txt sin imagen
    
    # Contar .txt vacíos vs .txt con contenido
    num_txt_vacios = 0
    num_txt_con_lineas = 0
    
    for lp in lbl_paths:
        with open(lp, "r") as f:
            lineas = [ln.strip() for ln in f.readlines() if ln.strip()]
        if len(lineas) == 0:
            num_txt_vacios += 1
        else:
            num_txt_con_lineas += 1
    
    print(f"Dataset verificado: {dir_name}")
    print(f"     Directorio imágenes: '{images_dir}'")
    print(f"     Directorio labels:   '{labels_dir}'")
    print(f"     Total imágenes:      {len(img_paths)}")
    print(f"     Total .txt labels:   {len(lbl_paths)}")
    print(f"     .txt vacíos:         {num_txt_vacios}")
    print(f"     .txt con anotaciones:{num_txt_con_lineas}")
    print(f"     Imágenes sin .txt:   {len(imgs_sin_label)}")
    print(f"     .txt sin imagen:     {len(labels_sin_img)}")


chequeo_consistencia("train")
chequeo_consistencia("valid")
chequeo_consistencia("test")

Con este resultado podemos verificar que:
* todas las imagenes (archivos jpg) tienen su label correspondiente (archivo txt - si una imagen no tiene ningún objecto detectado, el txt debe existir, pero vacío).
* todos los nombres de los archivos de inágenes corresponden a todos los nombres de los archivos de los labels.
* la cantidad de archivos en el dataset (para train, valid y test) coinciden a los datos cargados en los data loaders.

<br />

Verificamos las imágenes:

In [ ]:
def analizar_propiedades_imagenes(dir_name, max_print=10):
    dir_path = os.path.join(data_dir, dir_name)
    images_dir = os.path.join(dir_path, "images")

    img_paths = sorted(glob.glob(os.path.join(images_dir, "*")))
    if not img_paths:
        print(f"[{images_dir}] No se encontraron imágenes.")
        return

    combos = Counter()   # (width, height, channels, dtype)
    modos = Counter()    # img.mode (PIL)
    dtypes = Counter()   # np.array(img).dtype

    for path in img_paths:
        with Image.open(path) as img:
            w, h = img.size           # ancho, alto
            mode = img.mode           # p.ej. "L", "RGB", "RGBA", etc.
            arr = np.array(img)

            if arr.ndim == 2:
                canales = 1
            else:
                canales = arr.shape[2]

            dtype = arr.dtype

            modos[mode] += 1
            dtypes[dtype] += 1
            combos[(w, h, canales, dtype)] += 1

    print(f"Dataset verificado: {dir_name}")
    print(f"     Directorio: '{images_dir}'")
    print(f"     Total de imágenes: {len(img_paths)}")

    print("     Modos de color (PIL):")
    for m, c in modos.items():
        print(f"          {m}: {c}")

    print("     Tipos de dato (numpy dtypes):")
    for dt, c in dtypes.items():
        print(f"          {dt}: {c}")

    print("     Combinaciones (width, height, canales, dtype):")
    for (w, h, c, dt), count in combos.most_common(max_print):
        print(f"          {w}x{h}, canales={c}, dtype={dt}: {count} imágenes")

    if len(combos) == 1:
        print("     ✅ Todas las imágenes tienen el mismo tamaño, canales y dtype.")
    else:
        print("     ⚠️ Hay variabilidad en tamaño/canales/dtype entre las imágenes.")


analizar_propiedades_imagenes("train")
analizar_propiedades_imagenes("valid")
analizar_propiedades_imagenes("test")

Con este resultado podemos verificar que todas las imágenes tienen el mismo tamaño, canales y tipo.

Ahora generamos los histogramas de itensidad (RGB) de algunas imágenes:

In [ ]:
def mostrar_histogramas_intensidad(dir_name, num_samples):
    dir_path = os.path.join(data_dir, dir_name)
    images_dir = os.path.join(dir_path, "images")

    img_paths = sorted(glob.glob(os.path.join(images_dir, "*")))
    if not img_paths:
        print(f"[{images_dir}] No se encontraron imágenes.")
        return

    sample_paths = random.sample(img_paths, min(num_samples, len(img_paths)))

    print(f"Dataset: {dir_name}")
    print(f"     Directorio: '{images_dir}'")
    print(f"     Muestra de {len(sample_paths)} imágenes:\n")

    for path in sample_paths:
        with Image.open(path) as img:
            img = img.convert("RGB")
            arr = np.array(img)

        print(f"          Imagen: {os.path.basename(path)}")
        print(f"               Hape: {arr.shape}, dtype: {arr.dtype}")
        print(f"               Min: {arr.min()}, max: {arr.max()}, mean: {arr.mean():.2f}")

        # Histogramas por canal (R, G, B)
        canales = ["R", "G", "B"]
        plt.figure(figsize=(12, 3))
        plt.suptitle(f"Histogramas por canal - {os.path.basename(path)}")

        for c in range(3):
            plt.subplot(1, 3, c + 1)
            plt.hist(arr[..., c].flatten(), bins=128, range=(0, 255))
            plt.xlabel(f"Canal {canales[c]}")
            plt.ylabel("Frecuencia")

        plt.tight_layout()
        plt.show()


histograma_num_samples = 3

mostrar_histogramas_intensidad("train", num_samples=histograma_num_samples)
mostrar_histogramas_intensidad("valid", num_samples=histograma_num_samples)
mostrar_histogramas_intensidad("test",  num_samples=histograma_num_samples)

Acá sólo mostramos algunas pocas por grupo, pero revisamos varias más. Podemos observar que:

* La media es alta, en promedio los píxeles son bastantes claros (cercanos a la zona del blanco). Esto es común en imágenes de rayos X: gran parte es fondo claro y los objetos relevantes aparecen como estructuras más oscuras o de distinto tono.

* Los histogramas de R, G y B son muy parecidos. Esto indica que la imagen es pseudo-gris en RGB (mismo contenido replicado en los 3 canales o muy parecido). Para el modelo, esto es prácticamente una imagen en escala de grises.

* Esto no debería ser un problema: el modelo sigue teniendo información para distinguir objetos.

Ahora hacemos un análisis cuantitativo en todo el dataset del porcentaje de píxeles en 0 y 255:

In [ ]:
def porcentaje_pixeles_0_255(dir_name):
    dir_path = os.path.join(data_dir, dir_name)
    images_dir = os.path.join(dir_path, "images")
    img_paths = sorted(glob.glob(os.path.join(images_dir, "*")))
    if not img_paths:
        print(f"[{images_dir}] No se encontraron imágenes.")
        return

    total_pixels = 0
    total_zeros = 0
    total_255 = 0

    for path in img_paths:
        with Image.open(path) as img:
            img = img.convert("RGB")  # aseguramos 3 canales
            arr = np.array(img)

        # aplanamos todos los canales
        flat = arr.flatten()
        total_pixels += flat.size
        total_zeros += np.count_nonzero(flat == 0)
        total_255 += np.count_nonzero(flat == 255)

    pct_zeros = 100.0 * total_zeros / total_pixels
    pct_255 = 100.0 * total_255 / total_pixels

    print(f"Dataset: {dir_name}")
    print(f"     Directorio: '{images_dir}'")
    print(f"     Total de imágenes: {len(img_paths)}")
    print(f"     Total de píxeles: {total_pixels}")
    print(f"     Píxeles == 0  : {total_zeros} ({pct_zeros:.3f} %)")
    print(f"     Píxeles == 255: {total_255} ({pct_255:.3f} %)")


porcentaje_pixeles_0_255("train")
porcentaje_pixeles_0_255("valid")
porcentaje_pixeles_0_255("test")

En todo el dataset sólo alrededor del 0.046% de los píxeles son negro puro, mientras que aproxímadamente el 2.25% de los píxeles son blancoi puro, verificando lo que vimos en los hinstagramas.

Estos resultados indican que el dataset presenta un rango dinámico razonable, sin predominio excesivo de áreas completamente saturadas, aunque con un claro sesgo hacia intensidades altas debido al fondo del escáner.

---

#### Análisis de imágenes negativas (sin objetos) ####

In [ ]:
def contar_imagenes_sin_objetos(dl):
    total_imgs = 0
    sin_objetos = 0

    for imgs, targets in dl:              # targets = lista de dicts
        for t in targets:
            total_imgs += 1
            if t["boxes"].shape[0] == 0:  # 0 cajas -> imagen "negativa"
                sin_objetos += 1

    print("     Total de imágenes:", total_imgs)
    print("     Imágenes sin objetos:", sin_objetos)

print(f"Dataset: train")
contar_imagenes_sin_objetos(train_dataloader)
print(f"Dataset: valid")
contar_imagenes_sin_objetos(valid_dataloader)
print(f"Dataset: test")
contar_imagenes_sin_objetos(test_dataloader)

Como podemos ver no hay ninguna imagen sin objetos etiquetados: todas contienen al menos una caja asociada a alguna clase. El dataset está compuesto exclusivamente por ejemplos positivos, esto no permite evaluar ni entrenar el modelo ante equipaje sin amenazas, algo a tener en cuenta por su impacto en los falsos positivos.

#### Análisis de clases (frecuencia y balance)

In [ ]:
def get_distribution_from_dataset(dataset):
    labels_counter = Counter()

    # 1. Iteramos sobre la lista de imágenes que el dataset ya encontró
    for img_path in dataset.images:

        # 2. Recreamos la lógica del dataset para encontrar el archivo .txt
        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(dataset.labels_dir, base + ".txt")

        # 3. Usamos el método interno del dataset para leer las etiquetas
        #    Esto es mucho más rápido pues no carga la imagen.
        _boxes, labels_tensor = dataset._read_yolo_txt(lbl_path)

        # 4. Actualizamos el contador
        for class_id in labels_tensor.tolist():
            class_name = CLASS_NAMES[class_id]
            labels_counter.update([class_name])

    return labels_counter

print("Calculando distribuciones usando los objetos Dataset...")
train_counts = get_distribution_from_dataset(train_dataset)
valid_counts = get_distribution_from_dataset(valid_dataset)
test_counts = get_distribution_from_dataset(test_dataset)

dist_df = pd.DataFrame({
    'train': train_counts,
    'valid': valid_counts,
    'test': test_counts
})

dist_df = dist_df.fillna(0).astype(int)

dist_df.loc['Total'] = dist_df.sum()

print("Distribution de labels (counts):")
print(dist_df)

print("\nDistribution (porcentajes):")
percent_df = (dist_df.iloc[:-1] / dist_df.loc['Total']) * 100
print(percent_df.round(2))

Observamos un desbalance en la clase "pliers". Para eso podemos utilizar un Muestreo Ponderado.

En un `DataLoader` configurado con `shuffle=True`, el muestreo de datos se realiza de manera uniforme. Al comienzo de cada época de entrenamiento, el cargador genera una permutación aleatoria de los índices de todas las muestras disponibles en el dataset. Posteriormente, crea los lotes (batches) recorriendo secuencialmente esta lista de índices reordenados. Este método garantiza que cada muestra se utiliza exactamente una vez por época, en un orden aleato. Sin embargo, si el dataset sufre de un desbalance de clases significativo (como la baja representación de 'pliers'), este muestreo simplemente replicará dicho desbalance. El modelo observará las clases mayoritarias con una frecuencia proporcionalmente mayor, lo que introduce un sesgo en el aprendizaje y reduce su capacidad para generalizar sobre las clases minoritarias.

El `WeightedRandomSampler` se implementa para combatir activamente el desbalance de clases. En lugar de una selección uniforme, este método emplea un muestreo ponderado, donde la probabilidad de seleccionar cada muestra se define explícitamente. Para su configuración, se debe proveer un tensor de "pesos" que se corresponde uno a uno con cada muestra del dataset. Típicamente, a las muestras que contienen la clase minoritaria (ej. 'pliers') se les asigna un peso numérico mayor, mientras que las muestras de clases mayoritarias reciben un peso menor (ej. 1.0). El sampler utiliza estos pesos para construir una distribución de probabilidad. En la práctica, esto se implementa como un muestreo con reemplazo (replacement=True): las muestras con pesos altos tienen una mayor probabilidad de ser seleccionadas en cada extracción y, por tanto, pueden aparecer múltiples veces en una sola época. El resultado es un sobremuestreo (oversampling) efectivo de la clase minoritaria, presentando al modelo una distribución de clases más equilibrada durante el entrenamiento.



In [ ]:
PLIERS_CLASS_ID = CLASS_NAMES.index('pliers')

print("Calculando pesos para el sampler...")
weights = []

for img_path in train_dataset.images:
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(train_dataset.labels_dir, base + ".txt")
    _boxes, labels_tensor = train_dataset._read_yolo_txt(lbl_path)

    # Esta imagen CONTIENE 'pliers'?
    if PLIERS_CLASS_ID in labels_tensor:
        # Sí -> peso alto
        weights.append(2.0)
    else:
        # No -> peso normal
        weights.append(1.0)

print(f"Se crearon {len(weights)} pesos (uno por cada imagen de train).")

weights_tensor = torch.FloatTensor(weights)

sampler = WeightedRandomSampler(
    weights=weights_tensor,
    num_samples=len(weights_tensor), # Queremos un epoch del mismo tamaño
    replacement=True                 # Permite tomar la misma muestra más de una vez
)

print("Creando nuevo DataLoader con WeightedRandomSampler...")

# Nuevo dataloader utilizando weighted_random_sampler para apalear el problema de desbalance
train_dataloader_weighted_random_sampler = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=dataloader_batch_size,
    collate_fn=collate_fn,
    sampler=sampler,
    shuffle=False
)


print("¡Listo!")